In [33]:
#%pip install geopandas pyarrow folium matplotlib mapclassify branca -q

In [34]:
import geopandas as gpd
import pandas as pd

In [35]:
berk_zoning_gdf = gpd.read_parquet("../data/raw/berk_zoning.parquet") #Berkeley zoning data
mtc_sb79_zones_gdf = gpd.read_parquet("../data/raw/mtc_sb79_tod_zones.parquet") #MTC SB79 tier 1 and 2 zone data
berk_parcels_gdf = gpd.read_parquet("../data/raw/berk_parcels.parquet") #Berkeley parcel data

In [36]:
#initial row count
initial_count = len(berk_parcels_gdf)

#dissolve duplicate APN polygons into a single multi-part geometry per APN so each row is one parcel
#APNs refer to specific units, not parcels, so this makes each row one parcel
berk_parcels_gdf = berk_parcels_gdf.dissolve(by="apn", as_index=False)

#new row count
final_count = len(berk_parcels_gdf)

#difference in row counts (number of lost rows)
dissolved_count = initial_count - final_count

print(f"Initial rows: {initial_count}")
print(f"Final rows:   {final_count}")
print(f"Dissolved:    {dissolved_count} duplicate rows merged into existing APNs")

Initial rows: 29024
Final rows:   29003
Dissolved:    21 duplicate rows merged into existing APNs


In [37]:
def convert_crs(gdf_change, gdf_base):
    if gdf_change.crs != gdf_base.crs: #check if input gdf mathches base gdf
        gdf_change = gdf_change.to_crs(gdf_base.crs) #if they don't match, change the crs to match
    return gdf_change

In [38]:
#make all gdfs the same crs as the mtc gdf
gdfs = [berk_zoning_gdf, berk_parcels_gdf]
for gdf in gdfs:
    convert_crs(gdf, mtc_sb79_zones_gdf)

In [39]:
berk_parcels_gdf.columns

Index(['apn', 'geometry', ':id', ':version', ':created_at', ':updated_at',
       'situs_stree', 'date_updat', 'situs_str_1', 'situs_unit', 'situs_city',
       'situs_zip', 'use_code', 'building_ar', 'lot_size', 'situs_addre',
       'longitude', 'latitude', 'parcelid', 'ext_min_x', 'ext_min_y',
       'ext_max_x', 'ext_max_y'],
      dtype='str')

In [40]:
keep_parcel_columns = [
    'apn', #Assessor's Parcel Number
    'use_code', #Current land use (not what's zoned)
    'situs_addre', #Address
    'geometry'
]
berk_parcels_gdf = berk_parcels_gdf[keep_parcel_columns] #Keep only the columns necessary to this project to improve processeing speeds

In [41]:
def get_dominant_intersect(parcels_gdf, attribute_gdf):
    
    #create a temportary unique ID in case of duplicate parcels
    gdf = parcels_gdf.copy() #ensure not modifying parcels_gdf in-place
    gdf['temp_id'] = gdf.index

    before_count = len(parcels_gdf)

    #overaly the parcels with the attribute gdf their intersection
    intersected = gpd.overlay(gdf, attribute_gdf, how="intersection")
    
    duplicated_count = len(intersected)

    #get the area of each parcel - reproject to CA State Place Zone 3 (EPSG: 2227)
    #there are now duplicate parcels because they were assigned to multiple zones
    intersected["overlap_area"] = intersected.to_crs(epsg=2227).area

    #take only the parcel with the larger area in the SB79 zone
    dominant_intersect = (
        intersected.sort_values("overlap_area", ascending=False)
        .drop_duplicates(subset=["temp_id"])
    )

    matched_count = len(dominant_intersect)
    unmatched_count = before_count - matched_count
    
    # Print statement breakdown
    print(f"Initial parcels input:                 {before_count}")
    print(f"Total overlapping fragments created:    {duplicated_count}")
    print(f"Unique parcels matching attribute:     {matched_count}")
    print(f"Parcels with NO match (dropped):       {unmatched_count}\n")

    attr_cols = [
        col for col in dominant_intersect.columns 
        if col not in parcels_gdf.columns or col == "temp_id"
    ]
    
    return dominant_intersect[attr_cols]

In [42]:
berk_parcels = berk_parcels_gdf.copy()

#set the temp_id to compare the clipped versions to
berk_parcels["temp_id"] = berk_parcels.index

In [43]:
#overlay the parcels on the intersection of the SB79 zones and the Oakland zoning geometry
berk_parcel_tiers = get_dominant_intersect(berk_parcels, mtc_sb79_zones_gdf)
berk_parcel_zones = get_dominant_intersect(berk_parcels, berk_zoning_gdf)

Initial parcels input:                 29003
Total overlapping fragments created:    9833
Unique parcels matching attribute:     9318
Parcels with NO match (dropped):       19685

Initial parcels input:                 29003
Total overlapping fragments created:    32253
Unique parcels matching attribute:     28961
Parcels with NO match (dropped):       42



In [44]:
berk_parcel_zones

,temp_id,:id,:version,:created_at,:updated_at,zoneclass,zonedesc,baseelev,height,lastupdate,lasteditor,area,perimeter,blocks_id,new_block,zone,genplan,shape_area,shape_leng,overlap_area
14690,12731,row-iq44_wywk~mnxp,rv-b4zt_7ss9.rprz,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,R-5H,High Density Residential (Hillside Overlay),0.0,0.0,NaT,NaN,168455465.0,247231.306,619.0,060 2400,R,I,920294.116607,4679.27288863,9.901687e+06
25268,22116,row-bxb9~r9dc_fw5w,rv-ef7j-i7wa.uaz3,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,U,Unclassified,0.0,0.0,NaT,NaN,123201962.556,423306.722,0.0,060 2521,U,W,1199008.54248,13088.4202262,6.657880e+06
25243,22085,row-bxb9~r9dc_fw5w,rv-ef7j-i7wa.uaz3,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,U,Unclassified,0.0,0.0,NaT,NaN,123201962.556,423306.722,0.0,060 2521,U,W,1199008.54248,13088.4202262,1.936040e+06
49,41,row-bxb9~r9dc_fw5w,rv-ef7j-i7wa.uaz3,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,U,Unclassified,0.0,0.0,NaT,NaN,123201962.556,423306.722,0.0,060 2521,U,W,1199008.54248,13088.4202262,1.682938e+06
25247,22092,row-bxb9~r9dc_fw5w,rv-ef7j-i7wa.uaz3,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,U,Unclassified,0.0,0.0,NaT,NaN,123201962.556,423306.722,0.0,060 2521,U,W,1199008.54248,13088.4202262,1.297322e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,580,row-hcp5-xwdr~2ph7,rv-k5rc.b9y4~m4qs,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,C-AC,Adeline Corridor Commercial,0.0,0.0,NaT,NaN,14238344.431,298411.951,1020.0,055 1826,R,NC,165675.316283,10794.7155484,2.400824e-01
31544,28319,row-r6b9_5te7.ug5a,rv-tx7u.ykf9_39aj,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,R-1H,Single Family Residential (Hillside Overlay),0.0,0.0,NaT,NaN,2225069445.39,22833958.874,1.0,064 4295,R,OS,4985337.75292,189853.989082,8.205393e-02
31502,28278,row-r6b9_5te7.ug5a,rv-tx7u.ykf9_39aj,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,R-1H,Single Family Residential (Hillside Overlay),0.0,0.0,NaT,NaN,2225069445.39,22833958.874,1.0,064 4295,R,OS,4985337.75292,189853.989082,1.142633e-02
1,1,row-8cdh-yvp8.x6ae,rv-qa5t_g9rn-4r57,2026-08-12 19:46:34.275000+00:00,2026-08-12 19:46:34.275000+00:00,R-2A,Multi-Unit 2A,0.0,0.0,NaT,NaN,476417499.76,6640614.279,0.0,060 2453,R,OS,1690501.50041,88361.9884227,3.700538e-04


In [45]:
initial_count = len(berk_parcels)

#merge original parcels with parcel tiers
berk_parcels = berk_parcels.merge(
    berk_parcel_tiers,
    on="temp_id",
    how="inner" #drop parcels that don't have a tier match
)

tier_merge_count = len(berk_parcels)
print(f"Rows after merging SB79 Tiers:      {tier_merge_count}")
print(f"Change: {tier_merge_count - initial_count} rows")

#prevent duplicate column names
berk_parcels = berk_parcels.rename(columns={
    "zone_id": "tier_zone_id", 
    "zone_label": "tier_zone_label"
})

#keep only 3 relevant columns from the zonign data and temp_id to merge
berk_parcels = berk_parcels.merge(
    berk_parcel_zones[["temp_id", "zone", "zoneclass", "zonedesc", "genplan"]],
    on="temp_id",
    how="left" #keep all the parcels
)

zoning_merge_count = len(berk_parcels)
print(f"Rows after merging Oakland Zoning:  {zoning_merge_count}")
print(f"Change: {zoning_merge_count - tier_merge_count} rows")
print(f"Final Parcel count: {len(berk_parcels)}")

#drop temp_id because clipping and merging is done
berk_parcels = berk_parcels.drop(columns=["temp_id"])

Rows after merging SB79 Tiers:      9318
Change: -19685 rows
Rows after merging Oakland Zoning:  9318
Change: 0 rows
Final Parcel count: 9318


In [48]:
#save cleaned and joined gdf as a parqet file to use for analysis
berk_parcels.to_parquet("../data/processed/berk_parcels_sb79tiers_zones.parquet")